# 03 — Machine Learning Enzyme–Protein Screening

This notebook replaces the rule-based AMP scoring from notebook 02 with an ESM-2 language model classifier trained on curated AMP databases.

The goal is to identify protein–enzyme combinations that produce peptides with high ML-predicted antimicrobial probability, providing a more biologically grounded prioritization than the simple rule-based score.

The workflow:
1. Load or train the ESM-2 + logistic regression AMP classifier
2. Run screen_combinations_ml() across all whey proteins and enzymes
3. Rank combinations by ML-predicted AMP probability
4. Save top candidates for experimental validation

## Import HERALD ML Screening Workflow

This section imports the ML-based screening function and supporting modules.

The screen_combinations_ml() function combines:
* proteins.py — retrieves whey protein sequences from UniProt
* digestion.py — simulates enzymatic cleavage into peptide fragments
* predictor.py — computes ESM-2 embeddings and predicts AMP probability
* screening.py — summarizes ML scoring results across protein-enzyme combinations

The ESM-2 model and logistic regression classifier are loaded once here
and passed through the pipeline to avoid reloading on every peptide.

In [11]:
import sys
import os
import joblib
import esm
import torch
import pandas as pd

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

from herald.proteins import WHEY_PROTEINS
from herald.enzymes import ENZYME_RULES
from herald.databases import build_amp_database
from herald.predictor import prepare_dataset, compute_esm2_embeddings, train_classifier
from herald.screening import screen_combinations_ml

## Load ESM-2 Model and AMP Classifier

This section loads the two components required for ML-based AMP scoring:

**ESM-2** is a protein language model developed by Meta and trained on 250 million 
protein sequences. It converts each peptide sequence into a 320-dimensional embedding 
vector that captures biological properties such as charge distribution, hydrophobicity, 
and structural context. ESM-2 is used here as a fixed feature extractor and importantly, its weights 
are not updated during training.

**The AMP classifier** is a logistic regression model trained on top of ESM-2 embeddings 
using 17,897 validated antimicrobial peptides from the APD6 and DBAASP databases as 
positive examples, and 18,000 randomly generated peptide sequences as negative examples. 
The classifier achieved 97% accuracy and a ROC-AUC of 0.996 on the held-out test set.

If a cached classifier is found in `data/processed/amp_classifier.pkl`, it is loaded 
directly to avoid retraining. Otherwise, the full training pipeline is run and the 
classifier is saved for future use.

Both the ESM-2 model and classifier are loaded once here and passed through the 
screening pipeline since loading them per peptide would be prohibitively slow.

In [12]:
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()

PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
clf_path = os.path.join(PROCESSED_DIR, "amp_classifier.pkl")

if os.path.exists(clf_path):
    clf = joblib.load(clf_path)
    print("Loaded classifier from cache.")
else:
    amp_df = build_amp_database()
    X_train, X_test, y_train, y_test = prepare_dataset(amp_df, n_samples=18000)
    X_train_emb = compute_esm2_embeddings(X_train.tolist(), model, alphabet)
    clf = train_classifier(X_train_emb, y_train)
    print("Trained and saved new classifier.")

Loaded classifier from cache.


## Run ML-Based Protein–Enzyme Screening

This section runs the full ML screening pipeline across all whey proteins 
and enzymes defined in HERALD.

For each protein–enzyme combination, the workflow:
1. retrieves the protein sequence from the local UniProt cache,
2. simulates enzymatic digestion into candidate peptide fragments,
3. computes ESM-2 embeddings for each peptide,
4. predicts AMP probability using the trained logistic regression classifier, and
5. summarizes ML scoring metrics across all peptides for that combination.

Unlike the rule-based screening in notebook 02, which scored peptides using 
hand-crafted physicochemical heuristics, this notebook uses a classifier trained 
on 17,897 experimentally validated AMPs. The key output metric is amp_probability, the model's confidence that a given peptide is antimicrobial, rather than the 
simple_amp_score integer from the previous notebook.

The resulting table provides one row per protein–enzyme combination, ranked by 
the highest predicted AMP probability among all peptides produced by that combination.

In [13]:
results = screen_combinations_ml(
    proteins=WHEY_PROTEINS,
    enzymes=ENZYME_RULES,
    min_length=4,
    max_length=50,
    model=model,
    alphabet=alphabet,
    clf=clf,
)

results

,protein,accession_id,enzyme,min_length,max_length,num_peptides,num_predicted_amp,avg_probability,top_probability,top_peptide
0,beta-lactoglobulin,P02754,trypsin,4,50,13,9,0.549930,0.977336,IPAVFK
1,beta-lactoglobulin,P02754,chymotrypsin,4,50,19,11,0.540589,0.906039,ACQCL
2,alpha-lactalbumin,P00709,trypsin,4,50,12,7,0.557465,0.998233,FFVPLFLVGILFPAILAK
3,alpha-lactalbumin,P00709,chymotrypsin,4,50,16,7,0.479636,0.986193,GGIALPEL
4,lactoferrin,P24627,trypsin,4,50,58,39,0.642232,0.999670,LFVPALLSLGALGLCLAAPR
5,lactoferrin,P24627,chymotrypsin,4,50,67,42,0.617660,0.998732,AVAVVKKANEGL
6,BSA,P02769,trypsin,4,50,62,33,0.535034,0.993455,GACLLPK
7,BSA,P02769,chymotrypsin,4,50,61,31,0.522886,0.978912,AVSVL


## Rank Screening Results by ML-Predicted AMP Probability

The screening results are sorted by the highest predicted AMP probability 
observed across all peptides produced by each protein–enzyme combination.

Unlike the rule-based ranking in notebook 02, which used integer score thresholds, 
this ranking is based on the ESM-2 classifier's confidence that the top peptide 
from each combination is genuinely antimicrobial. A top_probability close to 1.0 
indicates the model is highly confident that at least one peptide produced by that 
combination has strong antimicrobial potential.

In [14]:
ranked = results.sort_values("top_probability", ascending=False).reset_index(drop=True)
ranked

,protein,accession_id,enzyme,min_length,max_length,num_peptides,num_predicted_amp,avg_probability,top_probability,top_peptide
0,lactoferrin,P24627,trypsin,4,50,58,39,0.642232,0.999670,LFVPALLSLGALGLCLAAPR
1,lactoferrin,P24627,chymotrypsin,4,50,67,42,0.617660,0.998732,AVAVVKKANEGL
2,alpha-lactalbumin,P00709,trypsin,4,50,12,7,0.557465,0.998233,FFVPLFLVGILFPAILAK
3,BSA,P02769,trypsin,4,50,62,33,0.535034,0.993455,GACLLPK
4,alpha-lactalbumin,P00709,chymotrypsin,4,50,16,7,0.479636,0.986193,GGIALPEL
5,BSA,P02769,chymotrypsin,4,50,61,31,0.522886,0.978912,AVSVL
6,beta-lactoglobulin,P02754,trypsin,4,50,13,9,0.549930,0.977336,IPAVFK
7,beta-lactoglobulin,P02754,chymotrypsin,4,50,19,11,0.540589,0.906039,ACQCL


## Top Candidate Peptides for Experimental Validation

This table summarizes the single highest-scoring peptide from each protein–enzyme 
combination. These candidates are the primary output of the computational pipeline 
and the starting point for Aim 2 wet lab validation.

Column descriptions:
- **top_peptide**: the peptide sequence recommended for synthesis and testing
- **top_probability**: ML confidence that this peptide is antimicrobial (0–1)
- **num_predicted_amp**: number of additional AMP-predicted peptides from this combination
- **avg_probability**: average AMP probability across all peptides from this combination

These predictions are based on sequence-derived ESM-2 embeddings and have not yet 
been experimentally validated. Candidates should be prioritized by top_probability 
but also considered in the context of synthesis feasibility, length, and the 
availability of the corresponding food-grade enzyme in the lab.

In [15]:
top_candidates = ranked[[
    "protein",
    "enzyme", 
    "top_peptide",
    "top_probability",
    "num_predicted_amp",
    "avg_probability",
]].copy()

top_candidates

,protein,enzyme,top_peptide,top_probability,num_predicted_amp,avg_probability
0,lactoferrin,trypsin,LFVPALLSLGALGLCLAAPR,0.999670,39,0.642232
1,lactoferrin,chymotrypsin,AVAVVKKANEGL,0.998732,42,0.617660
2,alpha-lactalbumin,trypsin,FFVPLFLVGILFPAILAK,0.998233,7,0.557465
3,BSA,trypsin,GACLLPK,0.993455,33,0.535034
4,alpha-lactalbumin,chymotrypsin,GGIALPEL,0.986193,7,0.479636
5,BSA,chymotrypsin,AVSVL,0.978912,31,0.522886
6,beta-lactoglobulin,trypsin,IPAVFK,0.977336,9,0.549930
7,beta-lactoglobulin,chymotrypsin,ACQCL,0.906039,11,0.540589


## Save ML Screening Results

The ranked screening results are saved to data/processed/ as a CSV file for 
downstream analysis, sharing with collaborators, and iterative refinement after 
experimental validation.

These results represent the computational output of Aim 1 and will be updated 
in Aim 3 as wet lab MIC values are incorporated to recalibrate the model.

In [ ]:
output_path = os.path.join(PROJECT_ROOT, "data", "processed", "ml_screening_results.csv")
ranked.to_csv(output_path, index=False)
print(f"Saved results to: {output_path}")